# 🎬 IMDB Sentiment Analysis
This notebook trains a simple text classification model to predict whether a movie review is **positive** or **negative**.

**Approach:**
- Convert text to numbers using **TF-IDF** (Term Frequency–Inverse Document Frequency)
- Classify using **Logistic Regression** — fast, interpretable, and effective for text
- Save the trained model as a `.pkl` file to use in the API

## Step 1: Import Libraries

In [ ]:
# Standard data libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn: for building and evaluating the model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# joblib: to save and load the trained model
import joblib

print('All libraries imported successfully!')

## Step 2: Load and Explore the Dataset

In [ ]:
# Load the IMDB dataset CSV
# The dataset has 2 columns: 'review' (text) and 'sentiment' (positive/negative)
df = pd.read_csv('IMDB_Dataset.csv')

print(f'Dataset shape: {df.shape}')   # rows x columns
print(f'\nFirst 3 rows:')
df.head(3)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

# Check label distribution
print('\nSentiment distribution:')
print(df['sentiment'].value_counts())

In [ ]:
# Visualize label balance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Sentiment Distribution', fontsize=14)
axes[0].set_ylabel('Number of Reviews')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Review length distribution
df['review_length'] = df['review'].apply(lambda x: len(x.split()))
df.groupby('sentiment')['review_length'].plot(kind='hist', bins=30, alpha=0.6,
    ax=axes[1], legend=True, color=['#e74c3c', '#2ecc71'])
axes[1].set_title('Review Length Distribution', fontsize=14)
axes[1].set_xlabel('Number of Words')
axes[1].legend(['Negative', 'Positive'])

plt.tight_layout()
plt.show()
print(f'Average review length: {df["review_length"].mean():.1f} words')

## Step 3: Prepare Data for Training

In [ ]:
# X = input features (the review text)
# y = target labels (positive / negative)
X = df['review']
y = df['sentiment']

# Split: 80% for training, 20% for testing
# random_state=42 ensures reproducibility (same split every time)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

## Step 4: Build the Model Pipeline

A **Pipeline** chains multiple steps together so they run in order:
1. **TF-IDF Vectorizer** — converts raw text into a numeric matrix
   - `max_features=5000`: keep only the top 5000 most informative words
   - `stop_words='english'`: ignore common words like 'the', 'is', 'at'
   - `ngram_range=(1,2)`: consider single words AND pairs (e.g. "not good")
2. **Logistic Regression** — a linear classifier, great for text data

In [ ]:
# Create pipeline with two steps
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,    # Use top 5000 words only
        stop_words='english', # Remove common English stopwords
        ngram_range=(1, 2)    # Use unigrams and bigrams
    )),
    ('clf', LogisticRegression(
        max_iter=1000         # Allow enough iterations to converge
    ))
])

print('Pipeline created:')
print(pipeline)

## Step 5: Train the Model

In [ ]:
# .fit() trains the model on the training data
# Internally: TF-IDF learns the vocabulary, then LR learns the weights
pipeline.fit(X_train, y_train)

print('Model training complete!')

## Step 6: Evaluate the Model

In [ ]:
# Predict on unseen test data
y_pred = pipeline.predict(X_test)

# Calculate accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f} ({acc*100:.1f}%)\n')

# Detailed report: precision, recall, F1 score per class
print('Classification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Visualize the Confusion Matrix
# Shows how many predictions were correct vs incorrect for each class
cm = confusion_matrix(y_test, y_pred, labels=['positive', 'negative'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Positive', 'Negative'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 7: Inspect the Most Influential Words

In [ ]:
# Extract the feature names (vocabulary) from TF-IDF
feature_names = pipeline.named_steps['tfidf'].get_feature_names_out()

# Extract the model coefficients (how much each word affects prediction)
# Positive coeff = word pushes toward 'positive' class
# Negative coeff = word pushes toward 'negative' class
coefs = pipeline.named_steps['clf'].coef_[0]

# Get top 15 positive and negative words
top_n = 15
top_pos_idx = coefs.argsort()[-top_n:][::-1]
top_neg_idx = coefs.argsort()[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Positive words
axes[0].barh(feature_names[top_pos_idx], coefs[top_pos_idx], color='#2ecc71')
axes[0].set_title('Top 15 Positive Words', fontsize=13)
axes[0].set_xlabel('Coefficient (weight)')
axes[0].invert_yaxis()

# Negative words
axes[1].barh(feature_names[top_neg_idx], coefs[top_neg_idx], color='#e74c3c')
axes[1].set_title('Top 15 Negative Words', fontsize=13)
axes[1].set_xlabel('Coefficient (weight)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Step 8: Test on Custom Reviews

In [ ]:
# Let's test our model on a few custom reviews
sample_reviews = [
    "This movie was absolutely amazing! I loved every moment of it.",
    "Terrible film. Boring plot and awful acting. Total waste of time.",
    "It was okay, nothing special but not bad either."
]

# predict() returns the class label
# predict_proba() returns the probability for each class
predictions = pipeline.predict(sample_reviews)
probabilities = pipeline.predict_proba(sample_reviews)

print('Custom Review Predictions:')
print('-' * 70)
for review, pred, prob in zip(sample_reviews, predictions, probabilities):
    classes = pipeline.classes_
    confidence = max(prob) * 100
    emoji = '😊' if pred == 'positive' else '😞'
    print(f'Review   : {review[:60]}...')
    print(f'Sentiment: {pred.upper()} {emoji} (confidence: {confidence:.1f}%)')
    print()

## Step 9: Save the Model as a .pkl File

In [ ]:
# Save the entire pipeline (TF-IDF + model) into one file
# This is what the FastAPI server will load to make predictions
model_path = 'sentiment_model.pkl'
joblib.dump(pipeline, model_path)

print(f'Model saved to: {model_path}')

# Verify we can reload it and it still works
loaded_model = joblib.load(model_path)
test_pred = loaded_model.predict(['Absolutely wonderful film, truly loved it!'])
print(f'Verification test: {test_pred[0]}')
print('Model file is ready for the FastAPI server!')